In [27]:
#from google.colab import drive
import os
import shutil

# 1. Kết nối với Google Drive (Sẽ hiện bảng hỏi quyền, cứ bấm Cho phép)
#drive.mount('/content/drive')

# 2. Tạo một thư mục vĩnh viễn trên Google Drive của bạn
#drive_path = '/content/drive/MyDrive/Result_V3'
#os.makedirs(drive_path, exist_ok=True)

# 3. Dọn dẹp thư mục ./result ảo của Colab (nếu đang có) để tránh xung đột
if os.path.islink('./result'):
    os.unlink('./result')
if os.path.islink('../result'):
    os.unlink('../result')

os.makedirs('./result', exist_ok=True)
os.makedirs('../result', exist_ok=True)

# 4. Tạo Symlink: Nối ../result thẳng vào Google Drive
#if not os.path.exists('./result'):
#    os.symlink(drive_path, './result')

#print("THÀNH CÔNG! Mọi file lưu vào ./result sẽ được chuyển thẳng lên Google Drive cá nhân của bạn.")

In [28]:
!nvidia-smi

Mon May 11 23:14:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [29]:
!pip install -q transformers datasets scikit-learn pandas matplotlib seaborn lime accelerate captum wordcloud

In [30]:
%%writefile dataset.py
# --- COPY TOÀN BỘ NỘI DUNG FILE dataset.py CỦA BẠN PASTE VÀO ĐÂY ---
import os
import pickle
import torch
import random
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text, remove_stop=False):
    text = text.lower()
    text = text.replace("@", "number")
    text = re.sub(r'[^\w\s]', '', text)

    if remove_stop:
        words = text.split()
        text = " ".join([w for w in words if w not in stop_words])

    return text

def load_and_tokenize_data(model_name="bert-base-uncased", augment=False):
    print("Đang tải dữ liệu từ Hugging Face Hub (armanc/pubmed-rct20k)...")
    ds = load_dataset("armanc/pubmed-rct20k")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    label_encoder = LabelEncoder()
    label_encoder.fit(ds['train']['label'])
    num_classes = len(label_encoder.classes_)
    print(f"Đã tìm thấy {num_classes} nhãn: {label_encoder.classes_}")

    os.makedirs('./result', exist_ok=True)
    with open('./result/label_encoder.pkl', 'wb') as f:
        pickle.dump(label_encoder, f)
    def preprocess_function(examples, is_train=True):
        should_remove_stop = "bert" not in model_name.lower()
        cleaned_texts = [clean_text(t, remove_stop=should_remove_stop) for t in examples['text']]
        if augment and is_train:
            aug_texts = []
            for t in cleaned_texts:
                words = t.split()
                if len(words) > 5:
                    words.pop(random.randint(0, len(words)-1))
                aug_texts.append(" ".join(words))
            cleaned_texts = aug_texts

        tokenized = tokenizer(cleaned_texts, padding="max_length", truncation=True, max_length=128)
        tokenized['labels'] = label_encoder.transform(examples['label']).tolist()
        return tokenized

    print("\nTiến hành tiền xử lý và Tokenize dữ liệu")
    train_dataset = ds['train'].map(lambda x: preprocess_function(x, is_train=True), batched=True, remove_columns=['label'])

    val_dataset = ds['validation'].map(lambda x: preprocess_function(x, is_train=False), batched=True, remove_columns=['label'])

    train_dataset.set_format(type="torch", columns=['input_ids', 'attention_mask', 'labels'], output_all_columns=True)
    val_dataset.set_format(type="torch", columns=['input_ids', 'attention_mask', 'labels'], output_all_columns=True)

    return train_dataset, val_dataset, num_classes, label_encoder, tokenizer

Overwriting dataset.py


In [31]:
%%writefile models.py
# --- COPY TOÀN BỘ NỘI DUNG FILE models.py CỦA BẠN PASTE VÀO ĐÂY ---
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from transformers.modeling_outputs import SequenceClassifierOutput

class BiLSTM_Model(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=32, num_classes=5, dropout=0.5):
        super(BiLSTM_Model, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        embedded = self.embedding(input_ids)

        lstm_out, _ = self.lstm(embedded)

        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).expand(lstm_out.size()).float()
            sum_embeddings = torch.sum(lstm_out * mask_expanded, 1)
            sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)
            pooled = sum_embeddings / sum_mask
        else:
            pooled = torch.mean(lstm_out, dim=1)

        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.fc.out_features), labels.view(-1))
        return SequenceClassifierOutput(loss=loss, logits=logits)

def build_model_pytorch(model_type, num_classes, vocab_size=None):
    if model_type == 'Transformer_BERT':
        print("Đang tải mô hình Transformer BERT")
        model = AutoModelForSequenceClassification.from_pretrained(
            "bert-base-uncased",
            num_labels=num_classes
        )
        return model

    elif model_type == 'Transformer_DistilBERT':
        print("Đang tải mô hình Transformer DistilBERT (Small)")
        model = AutoModelForSequenceClassification.from_pretrained(
            "distilbert-base-uncased",
            num_labels=num_classes
        )
        return model

    elif model_type == 'RNN_Bi-LSTM':
        print("Đang khởi tạo mạng RNN Bi-LSTM")
        if vocab_size is None:
            raise ValueError("Cần truyền vocab_size (kích thước từ điển) cho Bi-LSTM.")
        model = BiLSTM_Model(vocab_size=vocab_size, num_classes=num_classes)
        return model

    else:
        raise ValueError(f"Không hỗ trợ mô hình: {model_type}")

Overwriting models.py


In [32]:
%%writefile losses.py
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import torch.nn.functional as F

def get_class_weights(y_train):
    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    return torch.tensor(weights, dtype=torch.float)

def focal_loss_pytorch(gamma=2.0, weight=None):
    def focal_loss_fn(logits, labels):
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        p_t = torch.exp(-ce_loss)
        loss = (1 - p_t)**gamma * ce_loss
        if weight is not None:
            weight_device = weight.to(labels.device)
            alpha_t = weight_device.gather(0, labels)
            loss = alpha_t * loss

        return loss.mean()

    return focal_loss_fn

Overwriting losses.py


In [33]:
%%writefile evaluation.py
# --- COPY TOÀN BỘ NỘI DUNG FILE evaluation.py CỦA BẠN PASTE VÀO ĐÂY ---
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd
import os

def plot_training_history(log_history, scenario_name):
    train_epochs = [entry['epoch'] for entry in log_history if 'loss' in entry]
    train_loss = [entry['loss'] for entry in log_history if 'loss' in entry]
    val_epochs = [entry['epoch'] for entry in log_history if 'eval_loss' in entry]
    val_loss = [entry['eval_loss'] for entry in log_history if 'eval_loss' in entry]

    val_acc_epochs = [entry['epoch'] for entry in log_history if 'eval_accuracy' in entry]
    val_acc = [entry['eval_accuracy'] for entry in log_history if 'eval_accuracy' in entry]
    val_f1 = [entry.get('eval_f1', entry.get('eval_f1_macro', None)) for entry in log_history if 'eval_loss' in entry]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    if train_loss: ax1.plot(train_epochs, train_loss, label='Train Loss', marker='o')
    if val_loss: ax1.plot(val_epochs, val_loss, label='Val Loss', marker='s')
    ax1.set_title('Biểu đồ Loss')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    if val_acc:
        ax2.plot(val_acc_epochs, val_acc, label='Val Accuracy', color='green', marker='^')
    val_f1_clean = [f for f in val_f1 if f is not None]
    if val_f1_clean and len(val_f1_clean) == len(val_epochs):
        ax2.plot(val_epochs, val_f1_clean, label='Val F1-Score', color='orange', marker='d')

    ax2.set_title('Biểu đồ Metrics (Acc & F1)')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f'./result/{scenario_name}_training_history.png')
    plt.close()

def evaluate_model(y_true, y_pred, classes, scenario_name):
    print("Classification Report:\n")
    print(classification_report(y_true, y_pred, target_names=classes))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.xlabel('Dự đoán')
    plt.ylabel('Thực tế')
    plt.title(f'Confusion Matrix - {scenario_name}')

    plt.savefig(f'./result/{scenario_name}_confusion_matrix.png')
    plt.close()

def error_analysis(texts, y_true, y_pred_probs, label_encoder, scenario_name, top_n=5):
    y_pred_classes = np.argmax(y_pred_probs, axis=1)
    confidences = np.max(y_pred_probs, axis=1)
    errors_idx = np.where(y_pred_classes != y_true)[0]

    error_data = []
    for idx in errors_idx:
        py_idx = int(idx)
        error_data.append({
            'Text': texts[py_idx],
            'True_Label': label_encoder.inverse_transform([y_true[py_idx]])[0],
            'Predicted_Label': label_encoder.inverse_transform([y_pred_classes[py_idx]])[0],
            'Confidence': confidences[py_idx]
        })
    df_errors = pd.DataFrame(error_data)
    df_errors = df_errors.sort_values(by='Confidence', ascending=False)

    file_path = f'./result/{scenario_name}_error_analysis.csv'
    df_errors.to_csv(file_path, index=False)
    print(f"\nĐã xuất toàn bộ phân tích lỗi ra {file_path}")

def export_results_to_csv(y_true, y_pred, classes, model_name):
    report = classification_report(y_true, y_pred, target_names=classes, output_dict=True)
    df_report = pd.DataFrame(report).transpose()

    file_path = f"./result/{model_name}_classification_report.csv"
    df_report.to_csv(file_path)
    print(f"Đã xuất kết quả chi tiết ra: {file_path}")
    return df_report

def plot_comprehensive_comparison(results_list):
    """Vẽ biểu đồ so sánh Accuracy và F1 cho tất cả kịch bản"""
    df = pd.DataFrame(results_list)

    df['Accuracy_Num'] = df['Accuracy'].str.rstrip('%').astype('float')
    if 'F1-Score' in df.columns:
        df['F1_Num'] = df['F1-Score'].str.rstrip('%').astype('float')

    melted_df = pd.melt(df, id_vars=['Scenario'],
                        value_vars=['Accuracy_Num', 'F1_Num'] if 'F1-Score' in df.columns else ['Accuracy_Num'],
                        var_name='Metric', value_name='Score')

    plt.figure(figsize=(15, 7))
    sns.barplot(data=melted_df, x='Scenario', y='Score', hue='Metric', palette='Set2')
    plt.title('So sánh Accuracy và F1-Score giữa các mô hình')
    plt.ylim(0, 100)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Loại chỉ số')
    plt.tight_layout()
    plt.savefig('../result/Global_Comparison_Report.png')
    plt.close()
    print("Đã xuất biểu đồ so sánh tổng hợp (Accuracy & F1) tại ../result/Global_Comparison_Report.png")

Overwriting evaluation.py


In [34]:
%%writefile efficiency.py
# --- COPY TOÀN BỘ NỘI DUNG FILE efficiency.py CỦA BẠN PASTE VÀO ĐÂY ---\
import torch
import time
import os
import io
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def get_model_size(model):
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    size_mb = buffer.tell() / (1024 ** 2)
    return size_mb

def measure_inference_time_pytorch(model, dataloader, device="cpu"):
    model.to(device)
    model.eval()

    dummy_input = next(iter(dataloader))
    input_ids = dummy_input['input_ids'].to(device)
    attention_mask = dummy_input['attention_mask'].to(device)

    with torch.no_grad():
        for _ in range(10):
            _ = model(input_ids, attention_mask=attention_mask)

    start_time = time.time()
    with torch.no_grad():
        for batch in dataloader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            _ = model(ids, attention_mask=mask)
    end_time = time.time()

    avg_time = (end_time - start_time) / len(dataloader.dataset)
    return avg_time

def apply_quantization(model):
    quantized_model = torch.ao.quantization.quantize_dynamic(
        model,
        {torch.nn.Linear},
        dtype=torch.qint8
    )
    return quantized_model

def plot_efficiency_tradeoffs(results_list):
    if not results_list:
        return

    df = pd.DataFrame(results_list)

    df['Accuracy_Num'] = df['Accuracy'].str.rstrip('%').astype('float')
    df['Inference Time (s)'] = df['Inference Time (s)'].astype('float')
    df['Model Size (MB)'] = df['Model Size (MB)'].astype('float')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    sns.scatterplot(data=df, x='Inference Time (s)', y='Accuracy_Num', hue='Scenario', s=200, ax=ax1, palette='tab10')
    ax1.set_title('Trade-off: Tốc độ suy luận vs Độ chính xác')
    ax1.set_xlabel('Thời gian suy luận 1 mẫu (giây) -> Càng nhỏ càng tốt')
    ax1.set_ylabel('Độ chính xác (Accuracy %)')
    ax1.grid(True, linestyle='--', alpha=0.7)

    sns.scatterplot(data=df, x='Model Size (MB)', y='Accuracy_Num', hue='Scenario', s=200, ax=ax2, palette='tab10', marker='X')
    ax2.set_title('Trade-off: Kích thước mô hình vs Độ chính xác')
    ax2.set_xlabel('Kích thước mô hình (MB) -> Càng nhỏ càng tốt')
    ax2.set_ylabel('Độ chính xác (Accuracy %)')
    ax2.grid(True, linestyle='--', alpha=0.7)

    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.get_legend().remove()

    plt.tight_layout()
    plt.savefig('./result/Efficiency_Tradeoffs.png', bbox_inches='tight')
    plt.close()

Overwriting efficiency.py


In [35]:
%%writefile explain.py
# --- COPY TOÀN BỘ NỘI DUNG FILE explain.py CỦA BẠN PASTE VÀO ĐÂY ---
import numpy as np
import torch
import torch.nn.functional as F
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt
from captum.attr import LayerIntegratedGradients, visualization
import os

def explain_prediction_lime(text, model, tokenizer, label_encoder, scenario_name="model", device="cpu"):
    model.to(device)
    model.eval()
    class_names = list(label_encoder.classes_)
    explainer = LimeTextExplainer(class_names=class_names)

    def predictor(texts):
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=-1)

        return probs.cpu().numpy()

    exp = explainer.explain_instance(text, predictor, num_features=10, num_samples=500)

    fig = exp.as_pyplot_figure()
    plt.title(f"LIME - Case: {scenario_name}")
    plt.tight_layout()
    file_name = f'./result/lime_explanation_{scenario_name}.png'
    plt.savefig(file_name)
    plt.close()

def explain_prediction_captum(text, model, tokenizer, label_encoder, scenario_name, model_type, device="cpu"):
    model.to(device)
    if 'Bi-LSTM' in model_type:
        model.train()
        for m in model.modules():
            if isinstance(m, torch.nn.Dropout):
                m.eval()
    else:
        model.eval()

    for param in model.parameters():
        param.requires_grad = True

    def custom_forward(input_ids, attention_mask):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        probs = torch.nn.functional.softmax(logits, dim=-1)
        pred_class_idx = torch.argmax(probs, dim=-1).item()
        pred_prob = probs[0, pred_class_idx].item()

    def custom_forward(inputs_ids, mask):
        out = model(input_ids=inputs_ids, attention_mask=mask)
        return out.logits if hasattr(out, 'logits') else out[0]

    if 'DistilBERT' in model_type:
        embed_layer = model.distilbert.embeddings
    elif 'BERT' in model_type:
        embed_layer = model.bert.embeddings
    elif 'Bi-LSTM' in model_type:
        embed_layer = model.embedding
    else:
        raise ValueError("Không tìm thấy lớp Embedding cho mô hình này.")
    lig = LayerIntegratedGradients(custom_forward, embed_layer)

    ref_token_id = tokenizer.pad_token_id
    baselines = input_ids * 0 + ref_token_id

    attributions, delta = lig.attribute(
        inputs=input_ids,
        baselines=baselines,
        additional_forward_args=(attention_mask,),
        target=pred_class_idx,
        return_convergence_delta=True
    )

    attributions_sum = attributions.sum(dim=-1).squeeze(0)
    attributions_sum = attributions_sum / torch.norm(attributions_sum)

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    vis_record = visualization.VisualizationDataRecord(
        word_attributions=attributions_sum.cpu().detach().numpy(),
        pred_prob=pred_prob,
        pred_class=label_encoder.classes_[pred_class_idx],
        true_class="N/A", # Trong thực tế lúc test ta không cần show nhãn thật ở đây
        attr_class=label_encoder.classes_[pred_class_idx],
        attr_score=attributions_sum.sum().item(),
        raw_input_ids=tokens,
        convergence_score=delta.item()
    )

    html_content = visualization.visualize_text([vis_record])
    html_str = html_content.data

    file_path = f"./result/captum_explanation_{scenario_name}.html"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html_str)

Overwriting explain.py


In [36]:
%%writefile train.py
# --- COPY TOÀN BỘ NỘI DUNG FILE train.py CỦA BẠN PASTE VÀO ĐÂY ---
import numpy as np
import torch
import pandas as pd
import os
import time
from torch.utils.data import DataLoader
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Import các module custom
from dataset import load_and_tokenize_data
from models import build_model_pytorch
from evaluation import evaluate_model, export_results_to_csv, plot_training_history, error_analysis
from losses import focal_loss_pytorch, get_class_weights # <-- Thêm get_class_weights
from efficiency import get_model_size, measure_inference_time_pytorch, apply_quantization

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1_macro": f1,
        "precision": precision,
        "recall": recall
    }

def train_pipeline_pytorch(scenario_name, model_type='Transformer_BERT', epochs=5, use_focal_loss=False, freeze_backbone=False, use_augmentation=False):
    hf_model_name = "distilbert-base-uncased" if model_type == 'Transformer_DistilBERT' else "bert-base-uncased"

    train_dataset, val_dataset, num_classes, label_encoder, tokenizer = load_and_tokenize_data(
        model_name=hf_model_name,
        augment=use_augmentation
    )

    model = build_model_pytorch(model_type=model_type, num_classes=num_classes, vocab_size=tokenizer.vocab_size)

    if freeze_backbone:
        if model_type == 'Transformer_BERT':
            for param in model.bert.parameters():
                param.requires_grad = False
        elif model_type == 'Transformer_DistilBERT':
            for param in model.distilbert.parameters():
                param.requires_grad = False

    custom_loss_fn = None
    if use_focal_loss:
        train_labels = np.array(train_dataset['labels'])
        class_weights_tensor = get_class_weights(train_labels)
        custom_loss_fn = focal_loss_pytorch(gamma=2.0, weight=class_weights_tensor)

    class CustomTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels = inputs.get("labels")
            outputs = model(**inputs)
            logits = outputs.get("logits")
            if custom_loss_fn is not None:
                loss = custom_loss_fn(logits, labels)
            else:
                loss = outputs.loss if isinstance(outputs, dict) else outputs[0]
            return (loss, outputs) if return_outputs else loss

    lr = 2e-5 if 'Transformer' in model_type else 1e-3
    training_args = TrainingArguments(
        output_dir=f"./result/checkpoints_{scenario_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        fp16=torch.cuda.is_available(),
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        remove_unused_columns=True,
        disable_tqdm=True,
        logging_strategy="epoch"
    )

    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    return trainer, val_dataset, label_encoder, tokenizer

Overwriting train.py


In [37]:
%%writefile run.py
# --- COPY TOÀN BỘ NỘI DUNG FILE
import os
import time
import torch
import numpy as np
import pandas as pd
import gc
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

from train import train_pipeline_pytorch
from models import build_model_pytorch
from evaluation import (
    evaluate_model, export_results_to_csv, plot_training_history,
    error_analysis, plot_comprehensive_comparison
)
from efficiency import (
    get_model_size, measure_inference_time_pytorch, apply_quantization, plot_efficiency_tradeoffs
)
from explain import explain_prediction_lime, explain_prediction_captum

if __name__ == "__main__":
    os.makedirs('./result', exist_ok=True)

    scenarios = [
        {'name': 'BERT_Freeze', 'model': 'Transformer_BERT', 'freeze': True, 'focal': False, 'aug': False},
        {'name': 'BERT_Full', 'model': 'Transformer_BERT', 'freeze': False, 'focal': False, 'aug': False},
        {'name': 'Bi-LSTM_Full', 'model': 'RNN_Bi-LSTM', 'freeze': False, 'focal': False, 'aug': False},

        {'name': 'BERT_Full_Focal', 'model': 'Transformer_BERT', 'freeze': False, 'focal': True, 'aug': False},
        {'name': 'Bi-LSTM_Focal', 'model': 'RNN_Bi-LSTM', 'freeze': False, 'focal': True, 'aug': False},

        {'name': 'BERT_Augmented', 'model': 'Transformer_BERT', 'freeze': False, 'focal': False, 'aug': True},
        {'name': 'Bi-LSTM_Augmented', 'model': 'RNN_Bi-LSTM', 'freeze': False, 'focal': False, 'aug': True},

        {'name': 'DistilBERT_Small', 'model': 'Transformer_DistilBERT', 'freeze': False, 'focal': False, 'aug': False},
    ]

    summary_results = []
    current_device = "cuda" if torch.cuda.is_available() else "cpu"

    for sc in scenarios:
        print("\n" + "="*70)
        print(f"CASE ĐANG THỰC HIỆN CASE: {sc['name']} CASE")
        print("="*70)

        print(f"[HỆ THỐNG] Mô hình: {sc['model']}")
        if sc.get('freeze'): print("-> CASE Chế độ: Đóng băng Backbone (Feature Extraction).")
        if sc.get('focal'): print("-> CASE Chế độ: Sử dụng Focal Loss (Xử lý lệch nhãn).")
        if sc.get('aug'): print("-> CASE Chế độ: Sử dụng Data Augmentation.")

        start_train = time.time()

        trainer, val_dataset, label_encoder, tokenizer = train_pipeline_pytorch(
            scenario_name=sc['name'],
            model_type=sc['model'],
            epochs=5,
            use_focal_loss=sc['focal'],
            freeze_backbone=sc['freeze'],
            use_augmentation=sc['aug']
        )

        train_duration = time.time() - start_train
        print(f"-> [HOÀN TẤT] Thời gian huấn luyện: {train_duration:.1f} giây!")

        print("\n[HỆ THỐNG] Đang chạy đánh giá trên tập Validation...")
        predictions = trainer.predict(val_dataset)
        y_pred_logits = torch.from_numpy(predictions.predictions)
        y_pred_probs = torch.nn.functional.softmax(y_pred_logits, dim=-1).numpy()
        y_pred_classes = np.argmax(y_pred_probs, axis=-1)
        y_true = np.array(val_dataset['labels'])

        m_size = get_model_size(trainer.model)
        full_loader = DataLoader(val_dataset, batch_size=32)

        print(f"Đang đo tốc độ suy luận (Inference time) trên {current_device.upper()}...")
        inf_time = measure_inference_time_pytorch(trainer.model, full_loader, device=current_device)

        print("\n[HỆ THỐNG] Đang giải thích mô hình (XAI)...")
        sample_text = val_dataset['text'][0] # Lấy câu đầu tiên

        explain_prediction_lime(sample_text, trainer.model, tokenizer, label_encoder, scenario_name=sc['name'], device=current_device)
        print(f"-> Đã lưu biểu đồ LIME tại: ./result/lime_explanation_{sc['name']}.png")

        explain_prediction_captum(sample_text, trainer.model, tokenizer, label_encoder, scenario_name=sc['name'], model_type=sc['model'], device=current_device)
        print(f"-> Đã lưu nội soi Captum tại: ./result/captum_explanation_{sc['name']}.html")

        error_analysis(val_dataset['text'], y_true, y_pred_probs, label_encoder, scenario_name=sc['name'])
        print(f"-> Đã lưu file phân tích lỗi tại: ./result/{sc['name']}_error_analysis.csv")

        export_results_to_csv(y_true, y_pred_classes, label_encoder.classes_, sc['name'])
        print(f"-> Đã lưu Classification Report tại: ./result/{sc['name']}_classification_report.csv")

        evaluate_model(y_true, y_pred_classes, label_encoder.classes_, scenario_name=sc['name'])
        print(f"-> Đã lưu Confusion Matrix tại: ./result/{sc['name']}_confusion_matrix.png")

        plot_training_history(trainer.state.log_history, scenario_name=sc['name'])
        print(f"-> Đã lưu Biểu đồ Loss/Accuracy tại: ./result/{sc['name']}_training_history.png")

        acc = accuracy_score(y_true, y_pred_classes)
        f1 = f1_score(y_true, y_pred_classes, average='macro')

        summary_results.append({
            "Scenario": sc['name'],
            "Accuracy": f"{acc*100:.2f}%",
            "F1-Score": f"{f1*100:.2f}%",
            "Train Time (s)": f"{train_duration:.1f}",
            "Inference Time (s)": f"{inf_time:.4f}",
            "Model Size (MB)": f"{m_size:.2f}"
        })

        print("\nCASE Đang dọn dẹp VRAM để chuẩn bị cho CASE tiếp theo...")
        del trainer
        del predictions
        del y_pred_logits
        del full_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\n[HỆ THỐNG] Đang thực hiện Quantization (INT8) để minh họa Efficiency...")
    bert_base_model = build_model_pytorch('Transformer_BERT', num_classes=len(label_encoder.classes_), vocab_size=tokenizer.vocab_size)
    q_model = apply_quantization(bert_base_model)
    print(f"-> Kích thước BERT ban đầu: {get_model_size(bert_base_model):.2f} MB")
    print(f"-> Kích thước sau khi nén: {get_model_size(q_model):.2f} MB")

    print("\n[HỆ THỐNG] Đang vẽ các biểu đồ so sánh tổng hợp...")
    plot_efficiency_tradeoffs(summary_results)
    plot_comprehensive_comparison(summary_results)

    df_final = pd.DataFrame(summary_results)
    df_final.to_csv("../result/comprehensive_comparison_report.csv", index=False)

    print("\n" + "V" * 70)
    print(" HOÀN TẤT TOÀN BỘ QUY TRÌNH HUẤN LUYỆN VÀ ĐÁNH GIÁ 🎉")
    print(" KẾT QUẢ TỔNG HỢP NẰM TẠI: ../result/comprehensive_comparison_report.csv")
    print("V" * 70)

Overwriting run.py


In [38]:
!python run.py

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!

CASE ĐANG THỰC HIỆN CASE: BERT_Freeze CASE
[HỆ THỐNG] Mô hình: Transformer_BERT
-> CASE Chế độ: Đóng băng Backbone (Feature Extraction).
Đang tải dữ liệu từ Hugging Face Hub (armanc/pubmed-rct20k)...
Repo card metadata block was not found. Setting CardData to empty.
Đã tìm thấy 5 nhãn: ['background' 'conclusions' 'methods' 'objective' 'results']

Tiến hành tiền xử lý và Tokenize dữ liệu
Đang tải mô hình Transformer BERT
Loading weights: 100% 199/199 [00:00<00:00, 914.41it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPEC

In [39]:
import shutil
from google.colab import files

# Trỏ vào thư mục Local lưu kết quả mà chúng ta đã setup
local_path = '/content/result'

print("Đang nén toàn bộ thư mục từ Local Storage của Colab...")
# Nén toàn bộ thư mục /content/result thành file ket_qua_mohinh.zip
shutil.make_archive('ket_qua_mohinh', 'zip', local_path)

print("Đang chuẩn bị tải xuống máy tính của bạn...")
# Tải trực tiếp qua trình duyệt web
files.download('ket_qua_mohinh.zip')

print("🎉 Hoàn tất! Vui lòng không đóng trình duyệt cho đến khi file được tải xong.")

Đang nén toàn bộ thư mục từ Local Storage của Colab...
Đang chuẩn bị tải xuống máy tính của bạn...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Hoàn tất! Vui lòng không đóng trình duyệt cho đến khi file được tải xong.
